# Can we predict next-month account churn?

This notebook develops a leakage-aware answer using `data.csv` and `columns.csv`.

**Decision question:** can information available in the snapshot month rank accounts by their chance of cancelling in the following month?

**Evaluation design:** expanding-window, out-of-time validation. The last three months (2025-09 through 2025-11) are held back until one model is selected on earlier periods. This matches the intended future-scoring use case and avoids optimistic random row splits.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
DATA_PATH = Path("data.csv")
DICTIONARY_PATH = Path("columns.csv")

## 1. Load and validate the inputs

The assertions below make the analysis fail loudly if the schema, target, or account-month key is not as expected.

In [ ]:
dictionary = pd.read_csv(DICTIONARY_PATH)
raw = pd.read_csv(DATA_PATH)
df = raw.copy()
df["snapshot_month"] = pd.to_datetime(df["snapshot_month"], format="%Y-%m")

display(dictionary)

expected = set(dictionary["column"])
assert set(df.columns) == expected, "Data and dictionary columns differ"
assert df["churn_next_month"].isin([0, 1]).all(), "Target must be binary"
assert not df.duplicated(["account_ref", "snapshot_month"]).any(), "Duplicate account-month rows"
assert not df.isna().any().any(), "Unexpected missing values"

summary = pd.Series({
    "rows": len(df),
    "accounts": df["account_ref"].nunique(),
    "months": df["snapshot_month"].nunique(),
    "first snapshot": df["snapshot_month"].min().strftime("%Y-%m"),
    "last snapshot": df["snapshot_month"].max().strftime("%Y-%m"),
    "overall churn rate": df["churn_next_month"].mean(),
})
display(summary.to_frame("value"))

## 2. Leakage audit

`lifetime_months` is explicitly described as the account's **total** active lifetime from a 2026 CRM export. That information is not known in the snapshot month. It encodes when the account eventually ends and must not be used for a real prediction.

`account_ref` is also excluded: it is an identifier, not a reusable account attribute. `snapshot_month` is retained only for splitting and monitoring, not as a predictor. This avoids learning a one-off calendar churn spike that may not repeat.

In [ ]:
leak_rule = df["lifetime_months"].eq(df["tenure_months"])
leak_table = pd.crosstab(
    leak_rule.rename("lifetime == tenure"),
    df["churn_next_month"],
    margins=True,
)
display(leak_table)
print(
    "Leak rule precision:", f"{df.loc[leak_rule, 'churn_next_month'].mean():.1%}",
    "| recall:", f"{leak_rule[df['churn_next_month'].eq(1)].mean():.1%}",
)
assert (leak_rule.astype(int) == df["churn_next_month"]).all()
print("Result: the future-derived field reproduces the target exactly; it is excluded below.")

## 3. Target behavior and usable predictors

The monthly target rate is important context: abrupt changes make calibrated probabilities harder even when account ranking remains useful.

In [ ]:
monthly = (
    df.groupby("snapshot_month")["churn_next_month"]
      .agg(rows="size", churn_rate="mean")
      .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=monthly, x="snapshot_month", y="churn_rate", marker="o", ax=ax)
ax.axvspan(pd.Timestamp("2025-09-01"), pd.Timestamp("2025-11-30"), alpha=.12,
           color="tab:orange", label="final holdout")
ax.set(title="Next-month churn rate varies substantially over time",
       xlabel="Snapshot month", ylabel="Churn rate")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

display(monthly.assign(churn_rate=lambda x: x.churn_rate.map(lambda value: f"{value:.1%}")))

In [ ]:
usable_numeric = ["seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["usage_hours", "support_tickets", "tenure_months"]):
    sns.boxplot(data=df, x="churn_next_month", y=col, showfliers=False, ax=ax)
    ax.set(xlabel="Churn next month", title=col.replace("_", " ").title())
plt.tight_layout()
plt.show()

display(
    df.groupby("churn_next_month")[usable_numeric]
      .mean().T.rename(columns={0: "no churn mean", 1: "churn mean"})
      .round(2)
)

## 4. Out-of-time model comparison

Two development folds emulate retraining on all prior data and scoring the next three months. The final three months remain untouched during model choice.

Metrics:

- **ROC AUC:** ranking across churners and non-churners; 0.5 is random.
- **Average precision (AP):** ranking quality focused on the minority churn class; the no-skill reference is the churn rate.
- **Log loss / Brier score:** probability quality; lower is better.
- **Top-20% lift:** churn rate among the highest-risk 20%, divided by the overall rate. This corresponds to a retention team with capacity to contact one fifth of accounts.

In [ ]:
target = "churn_next_month"
features = ["plan", "seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]
numeric_features = ["seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]
categorical_features = ["plan"]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

models = {
    "prior baseline": DummyClassifier(strategy="prior"),
    "logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=15, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

dev_folds = [
    ("2025-03", "2025-05"),
    ("2025-06", "2025-08"),
]

def top_fraction_metrics(y, score, fraction=0.20):
    y = np.asarray(y)
    score = np.asarray(score)
    n_top = max(1, int(np.ceil(len(y) * fraction)))
    top = np.argsort(-score)[:n_top]
    precision = y[top].mean()
    recall = y[top].sum() / y.sum()
    return precision, recall, precision / y.mean()

def metric_row(y, score):
    precision, recall, lift = top_fraction_metrics(y, score)
    return {
        "n": len(y),
        "prevalence": np.mean(y),
        "roc_auc": roc_auc_score(y, score),
        "average_precision": average_precision_score(y, score),
        "log_loss": log_loss(y, score, labels=[0, 1]),
        "brier": brier_score_loss(y, score),
        "top20_precision": precision,
        "top20_recall": recall,
        "top20_lift": lift,
        "mean_prediction": np.mean(score),
    }

dev_rows = []
for model_name, estimator in models.items():
    for start, end in dev_folds:
        start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
        train = df[df["snapshot_month"] < start_ts]
        valid = df[df["snapshot_month"].between(start_ts, end_ts)]
        pipe = Pipeline([("preprocess", clone(preprocessor)), ("model", clone(estimator))])
        pipe.fit(train[features], train[target])
        score = pipe.predict_proba(valid[features])[:, 1]
        dev_rows.append({"model": model_name, "validation_period": f"{start} to {end}",
                         **metric_row(valid[target], score)})

dev_results = pd.DataFrame(dev_rows)
display(dev_results.round(3))
display(
    dev_results.groupby("model")[["roc_auc", "average_precision", "log_loss", "top20_lift"]]
               .mean().sort_values("roc_auc", ascending=False).round(3)
)

The logistic model is selected before opening the final holdout: it has the best mean ROC AUC and is simpler and more stable than the forest. The baseline's constant scores have no ranking value.

## 5. One-time final holdout evaluation

Train through 2025-08 and evaluate on 2025-09 through 2025-11. The same account may occur in earlier months, which is intentional: in deployment, prior snapshots of an active account are available. No identifier or prior target is supplied to the model.

In [ ]:
holdout_start = pd.Timestamp("2025-09-01")
train = df[df["snapshot_month"] < holdout_start]
test = df[df["snapshot_month"] >= holdout_start].copy()

selected_model = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
selected_model.fit(train[features], train[target])
test["predicted_risk"] = selected_model.predict_proba(test[features])[:, 1]

holdout_metrics = pd.Series(metric_row(test[target], test["predicted_risk"]), name="final holdout")
display(holdout_metrics.to_frame().round(3))

# Capacity-based decision rule: flag exactly the highest-risk 20%.
n_flag = int(np.ceil(0.20 * len(test)))
flagged_index = test["predicted_risk"].nlargest(n_flag).index
test["flag_top20"] = test.index.isin(flagged_index)
cm = confusion_matrix(test[target], test["flag_top20"])
display(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["not flagged", "flagged"]))

### Uncertainty via account-cluster bootstrap

Rows from the same account are correlated. The bootstrap therefore resamples whole accounts, not individual rows. These intervals describe sampling uncertainty in this holdout; they do not protect against a new regime shift.

In [ ]:
def cluster_bootstrap_metrics(frame, n_boot=500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    groups = {k: g for k, g in frame.groupby("account_ref")}
    account_ids = np.array(list(groups))
    rows = []
    for _ in range(n_boot):
        sampled_ids = rng.choice(account_ids, size=len(account_ids), replace=True)
        sample = pd.concat([groups[a] for a in sampled_ids], ignore_index=True)
        y, score = sample[target].to_numpy(), sample["predicted_risk"].to_numpy()
        if np.unique(y).size < 2:
            continue
        m = metric_row(y, score)
        rows.append({k: m[k] for k in ["roc_auc", "average_precision", "top20_lift"]})
    return pd.DataFrame(rows)

boot = cluster_bootstrap_metrics(test)
intervals = pd.DataFrame({
    "estimate": holdout_metrics[["roc_auc", "average_precision", "top20_lift"]],
    "95% low": boot.quantile(.025),
    "95% high": boot.quantile(.975),
})
display(intervals.round(3))

In [ ]:
# Ranking and calibration diagnostics on the final holdout.
test["risk_decile"] = pd.qcut(
    test["predicted_risk"].rank(method="first"), 10, labels=False
) + 1
deciles = (
    test.groupby("risk_decile")
        .agg(accounts=(target, "size"), predicted=("predicted_risk", "mean"),
             observed=(target, "mean"))
        .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=deciles, x="risk_decile", y="observed", color="tab:blue", ax=axes[0])
axes[0].set(title="Observed churn rises across predicted-risk deciles",
            xlabel="Predicted-risk decile (10 = highest)", ylabel="Observed churn")
axes[0].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

axes[1].plot([0, .4], [0, .4], "--", color="gray", label="perfect calibration")
axes[1].scatter(deciles["predicted"], deciles["observed"], s=55)
axes[1].set(title="Calibration by risk decile", xlabel="Mean predicted probability",
            ylabel="Observed churn", xlim=(0, .4), ylim=(0, .4))
axes[1].legend()
plt.tight_layout()
plt.show()

display(deciles.round(3))

## 6. What drives the model?

Standardized logistic coefficients show associations used for ranking; they are **not causal effects**. One-hot plan coefficients are relative to the omitted/intercept structure of the encoded model.

In [ ]:
feature_names = selected_model.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(
    selected_model.named_steps["model"].coef_[0], index=feature_names, name="coefficient"
).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = np.where(coefs > 0, "tab:red", "tab:blue")
coefs.plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Logistic regression coefficients", xlabel="Coefficient (standardized numeric inputs)")
plt.tight_layout()
plt.show()
display(coefs.to_frame().round(3))

## 7. Conclusion

The cell below turns the untouched-holdout results into the answer. A model is considered operationally useful here if it beats random ranking and concentrates materially more churners into a feasible review group. Probability estimates require a stricter standard because the underlying monthly rate changed sharply.

In [ ]:
auc = holdout_metrics["roc_auc"]
ap = holdout_metrics["average_precision"]
prev = holdout_metrics["prevalence"]
lift = holdout_metrics["top20_lift"]
recall = holdout_metrics["top20_recall"]
pred_mean = holdout_metrics["mean_prediction"]

print("YES — there is useful, moderate signal for prioritizing accounts, but not yet a fully reliable probability model.")
print()
print(f"• Final out-of-time ROC AUC: {auc:.3f} (random = 0.500).")
print(f"• Average precision: {ap:.3f}, versus {prev:.3f} for a no-skill ranking.")
print(f"• The highest-risk 20% contain {recall:.1%} of churners and have {lift:.2f}× the holdout's churn rate.")
print(f"• Mean predicted risk is {pred_mean:.1%}, while observed churn is {prev:.1%}; probabilities are not well calibrated.")
print()
print("Recommended use: pilot it as a ranking tool for retention outreach, measure incremental saves in a randomized test, and monitor monthly AUC, lift, prevalence, and calibration.")
print("Do not use lifetime_months. Before production, confirm feature availability at scoring time, investigate the 2024-10 to 2025-02 churn spike, and retrain/recalibrate on a regular cadence.")